# ESM-2 embedding extraction on Colab T4

Resumes the run from the checkpoint produced on the M2. Same model + `sample_n` so the existing rows in `data/embeddings.jsonl` are byte-compatible. Only `--batch-size` changes (8 → 64) to use the T4 properly.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have on your laptop:
   - `data/embeddings.jsonl` (the resume checkpoint)
   - `data/bacdive_phenotypes.parquet` (the strain list)
3. A GitHub Personal Access Token with `repo` scope (the repo is private)
4. Your `NCBI_API_KEY`

Estimated wall-clock: **1–3 hr** for the remaining ~15K genomes on T4.

**Recommended: enable Step 5 (Drive durability).** Without it, a Colab disconnect wipes everything since the last manual download. With it, the JSONL lives in your Drive and survives session loss — just rerun the cells next time and it picks up where it left off.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo (private, needs PAT)

In [ ]:
from getpass import getpass
import os, subprocess

pat = getpass('GitHub PAT (with repo scope): ')
url = f'https://{pat}@github.com/miyu-horiuchi/microbe-model.git'
subprocess.run(['git', 'clone', url], check=True)
os.chdir('microbe-model')
del pat, url  # don't keep the token around
!git log --oneline -3

## 3. Install (~3 min)

In [ ]:
!pip install -q -e ".[embeddings]"

## 4. Upload the checkpoint files

Pick **both** `data/embeddings.jsonl` (~13 MB) and `data/bacdive_phenotypes.parquet` (~1.6 MB) when the file picker opens.

If you're rerunning after a disconnect and you've already done Step 5 once, you can skip this cell — Drive already has your latest jsonl. But you still need the parquet, so the easiest path is to upload both every time.

In [ ]:
from google.colab import files
import os, shutil

os.makedirs('data', exist_ok=True)
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, os.path.join('data', fname))
!ls -la data/

## 5. Mount Google Drive for durability *(strongly recommended)*

Mounts your Drive at `/content/drive`, then **symlinks** `data/embeddings.jsonl` to a file inside Drive. The extraction script writes to that path unchanged — but the data physically lives in your Drive.

What this buys you:
- Colab disconnects → your laptop sleeps → your browser closes: doesn't matter. The jsonl keeps whatever was already flushed.
- Next session: just rerun all the cells. Step 5 detects the existing Drive file and reuses it; the extraction script skips genomes already done.

First run will pop a Google auth flow — sign in with the Google account that owns the Drive.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/microbe-model-embeddings'
os.makedirs(DRIVE_DIR, exist_ok=True)

drive_jsonl = f'{DRIVE_DIR}/embeddings.jsonl'
local_jsonl = 'data/embeddings.jsonl'

# Seed Drive from the just-uploaded local file IF Drive doesn't already have a (longer) checkpoint.
def _rows(path):
    return sum(1 for _ in open(path)) if os.path.exists(path) else 0

drive_rows = _rows(drive_jsonl)
local_rows = _rows(local_jsonl)
print(f'rows in Drive: {drive_rows:,}  |  rows in local upload: {local_rows:,}')

if local_rows > drive_rows:
    shutil.copy(local_jsonl, drive_jsonl)
    print(f'Local upload was ahead — copied {local_rows:,} rows to Drive.')
elif drive_rows > 0:
    print(f'Drive checkpoint is current ({drive_rows:,} rows). Reusing it.')
else:
    open(drive_jsonl, 'a').close()
    print('No checkpoint anywhere — starting fresh on Drive.')

# Replace the local file with a symlink pointing into Drive.
if os.path.lexists(local_jsonl):
    os.remove(local_jsonl)
os.symlink(drive_jsonl, local_jsonl)

print(f'\ndata/embeddings.jsonl -> {drive_jsonl}')
!ls -la data/embeddings.jsonl

## 6. Set NCBI API key

In [ ]:
import os
from getpass import getpass
os.environ['NCBI_API_KEY'] = getpass('NCBI_API_KEY: ')
# write to .env so the script's config loader picks it up too
with open('.env', 'w') as f:
    f.write(f"NCBI_API_KEY={os.environ['NCBI_API_KEY']}\n")

## 7. Keep the session alive

Colab disconnects after ~90 min of UI inactivity. Run this in your browser console (F12 → Console) before walking away — it clicks the connect button every minute:

```js
setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
```

If you've enabled Step 5, this is less critical — even a disconnect doesn't cost you progress. But it still helps avoid having to manually rerun the cells.

## 8. Run extraction

Same model + `sample_n` as the M2 run. `batch_size=64` to use T4 properly (M2 was at 8). Resumable — reads `data/embeddings.jsonl`, skips genomes already done.

In [ ]:
!python scripts/11_extract_embeddings.py \
    --model facebook/esm2_t6_8M_UR50D \
    --sample-n 20 \
    --batch-size 64

## 9. Download results

If Step 5 is enabled, the JSONL is already in your Drive at `MyDrive/microbe-model-embeddings/embeddings.jsonl` — you can pull it from drive.google.com directly. The parquet is still local; the cell below also copies it into Drive for safekeeping, then offers both as browser downloads.

In [ ]:
import os, shutil
from google.colab import files

DRIVE_DIR = '/content/drive/MyDrive/microbe-model-embeddings'
if os.path.isdir('/content/drive/MyDrive') and os.path.exists('data/embeddings.parquet'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy('data/embeddings.parquet', f'{DRIVE_DIR}/embeddings.parquet')
    print(f'Mirrored parquet to {DRIVE_DIR}/embeddings.parquet')

files.download('data/embeddings.jsonl')
files.download('data/embeddings.parquet')